# baseline_TCN (Source-only) — accuracy + DTW(vs samsung1)

Source(samsung1)로만 학습된 `weights/tcn_seed42_baseline_best.pth` 를 그대로 불러와,
Target 디바이스 데이터 **전체**에 대한 윈도우 정확도와 samsung1 IMU 와의 DTW 를 측정한다.

모든 로직은 `eval_utils.py` 로 분리했고, 이 노트북은 *데이터를 로드해 넘기고 결과를 표로 정리*만 한다.
→ parquet 경로가 아니라 **로드된 DataFrame** 을 넘기므로, 같은 함수로 48가지 IMU 축
순열·부호 탐색까지 그대로 재사용할 수 있다.

- `data/samsung2_YXZ.parquet`          : IMU 축이 samsung1 기준으로 정렬(YXZ 재라벨링)된 버전
- `data/samsung2.parquet` : `triceps_X` ↔ `triceps_Y` 를 되돌린 원본(축 미정렬)

In [ ]:
import os, sys
import pandas as pd

ROOT = os.path.abspath('../..')
sys.path.insert(0, ROOT)
import eval_utils as U

# Source-only 사전학습 모델 + samsung1 DTW reference (48회 재사용하므로 1회만 빌드)
model = U.load_baseline_model(os.path.join(ROOT, 'weights', 'tcn_seed42_baseline_best.pth'))
df_src = pd.read_parquet(os.path.join(ROOT, 'data', 'samsung1.parquet'))
dtw_ref = U.build_dtw_reference(df_src)
print('model + reference ready | device:', next(model.parameters()).device)

In [ ]:
# 평가할 Target parquet 들 — 로드해서 DataFrame 으로 넘긴다
TARGETS = {
    'samsung2 (aligned)': 'samsung2_YXZ.parquet',
    'samsung2_original':  'samsung2.parquet',
}

results = {}
for name, fname in TARGETS.items():
    df_tgt = pd.read_parquet(os.path.join(ROOT, 'data', fname))
    results[name] = U.evaluate(df_tgt, model, dtw_ref)   # {accuracy, dtw, n_windows}
    print(f'[{name}] {results[name]}')

In [ ]:
# 요약
print('=' * 64)
print(f'{"dataset":<22}{"#windows":>10}{"accuracy":>12}{"DTW(vs s1)":>16}')
print('-' * 64)
for name, r in results.items():
    print(f'{name:<22}{r["n_windows"]:>10}{r["accuracy"]:>11.2f}%{r["dtw"]:>16.2f}')
print('=' * 64)